# Rotina principal do experimento

TODO:

In [2]:
%run funcoes.py

Carregando dataset 'cardiffnlp/tweet_topic_single'...
4374 documentos carregados.
Baixando stopwords do NLTK...
Inicializando cliente LLM para o host '127.0.0.1:11434' e modelo 'llama3.1'...
Carregando modelo de embedding 'all-MiniLM-L6-v2'...
Gerando embeddings para 4374 textos...


Batches: 100%|██████████| 137/137 [00:01<00:00, 77.04it/s] 


Construindo índice FAISS para 4374 vetores de dimensão 384...
Índice construído com sucesso.

--- INICIANDO PROCESSO DE RAG PARA 500 AMOSTRAS ---

==================== Processando Amostra 1/500 ====================

Enviando documento para o LLM:
'I love you #SEVENTEEN you icons  #세븐틴 this won’t make any sense #헹가래 ok my kings #Henggarae will win...'
LLM respondeu com o tópico inicial: 'Music'
Gerando embeddings para 1 textos...
Buscando 3 documentos similares para a consulta...
Enviando contexto para refinar o tópico 'Music'...
LLM refinou para o tópico: 'K-Pop'

--- RESULTADO DA AMOSTRA ---
Documento Original: 'I love you #SEVENTEEN you icons  #세븐틴 this won’t make any sense #헹가래 ok my kings #Henggarae will win awards #Left_n_Right will win soty and {@SEVENTEEN@} are the bestest boys...'
Tópico Inicial Gerado: Music
Tópico Refinado com RAG: K-Pop

==================== Processando Amostra 2/500 ====================

Enviando documento para o LLM:
'Picking up some good stuff (@ Joe - {{

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Coerência calculada: 0.8698492146504357
Calculando Topic Diversity...
Diversidade calculada: 0.6148525664361121

Resultados salvos com sucesso em 'llm_topic_results.csv'
--- EXPERIMENTO CONCLUÍDO ---


#### Rodando LDA

In [11]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
import gensim
from gensim import corpora, models
from gensim.models import CoherenceModel
from gensim.utils import simple_preprocess
from collections import Counter, defaultdict
import re
import numpy as np

In [12]:
# --- 1. Carregar o CSV Gerado pelo LLM ---
CSV_FILENAME = "llm_topic_results.csv"
try:
    df = pd.read_csv(CSV_FILENAME)
    print(f"{len(df)} documentos carregados do arquivo '{CSV_FILENAME}'.")
except FileNotFoundError:
    print(f"ERRO: Arquivo '{CSV_FILENAME}' não encontrado. Você rodou o 'funcoes.py' primeiro?")
    # Se der erro aqui, não continue

# --- 2. Setup de Stopwords (IDÊNTICO AO funcoes.py) ---
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
print("Stopwords carregadas.")


# --- 3. Funções Helper (IDÊNTICAS AO funcoes.py) ---

def preprocess_text(text: str, stop_words: set) -> list[str]:
    """
    Tokeniza, remove stopwords, pontuação e palavras curtas.
    Esta função DEVE ser idêntica à usada no script do LLM.
    """
    # simple_preprocess faz a tokenização e passa para minúsculo
    return [word for word in simple_preprocess(text) if word not in stop_words and len(word) > 2]

def calculate_topic_coherence(topics_with_words_list: list[list[str]], 
                              documents_processed: list[list[str]], 
                              dictionary: corpora.Dictionary, 
                              coherence_type: str = 'c_v') -> float:
    """
    Calcula a coerência (ex: C_v) para um conjunto de tópicos do LDA.
    """
    print(f"Calculando Topic Coherence ({coherence_type}) para o LDA...")
    if not topics_with_words_list:
        print("Lista de tópicos vazia.")
        return 0.0
    
    try:
        coherence_model = CoherenceModel(
            topics=topics_with_words_list,
            texts=documents_processed,
            dictionary=dictionary,
            coherence=coherence_type
        )
        coherence = coherence_model.get_coherence()
        print(f"Coerência LDA (C_v) calculada: {coherence}")
        return coherence
    except Exception as e:
        print(f"Erro ao calcular coerência do LDA: {e}")
        return 0.0

def calculate_topic_diversity(topics_with_words_list: list[list[str]]) -> float:
    """
    Calcula a diversidade (proporção de palavras únicas) para o LDA.
    """
    print("Calculando Topic Diversity para o LDA...")
    if not topics_with_words_list:
        return 0.0
    
    # Pega as top N palavras de cada tópico
    all_words = [word for topic in topics_with_words_list for word in topic]
    if not all_words:
        return 0.0
    
    unique_words = set(all_words)
    diversity = len(unique_words) / len(all_words)
    print(f"Diversidade LDA calculada: {diversity}")
    return diversity

print("\nFunções de pré-processamento e métricas prontas.")
df.head(2)

500 documentos carregados do arquivo 'llm_topic_results.csv'.
Stopwords carregadas.

Funções de pré-processamento e métricas prontas.


,documento,classificação 1,classificação 2,topic_coherence,topic_diversity
0,I love you #SEVENTEEN you icons #세븐틴 this won...,Music,K-Pop,0.869849,0.614853
1,Picking up some good stuff (@ Joe - {{USERNAME...,Shopping,Shopping,0.869849,0.614853


In [13]:
print("Iniciando pré-processamento dos documentos para o LDA...")

# 1. Aplicar a função de pré-processamento na coluna 'documento'
# Usamos .astype(str) para garantir que mesmo que haja um 'None' ou 'NaN', ele não quebre
processed_docs = df['documento'].astype(str).apply(lambda x: preprocess_text(x, stop_words)).tolist()

# 2. Criar Dicionário (id -> palavra)
dictionary = corpora.Dictionary(processed_docs)

# 3. Criar Corpus (Bag-of-Words: (id_palavra, frequencia))
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

print(f"Processamento concluído.")
print(f"Tamanho do Dicionário: {len(dictionary)} palavras únicas.")
print(f"Tamanho do Corpus: {len(corpus)} documentos.")

Iniciando pré-processamento dos documentos para o LDA...
Processamento concluído.
Tamanho do Dicionário: 3733 palavras únicas.
Tamanho do Corpus: 500 documentos.


In [14]:
# --- Treinamento do Modelo LDA ---

# 1. Determinar o número de tópicos (k)
# Para uma comparação justa, vamos usar o número de tópicos únicos 
# que o LLM (classificação 2) encontrou.

# Filtra 'ERRO' e conta tópicos únicos
llm_topics_validos = df[~df['classificação 2'].str.contains("ERRO", na=False)]['classificação 2'].unique()
NUM_TOPICOS_LDA = len(llm_topics_validos)

# Se a amostra for muito pequena (ex: 5) ou só der erro, 
# o número pode ser 0 ou 1. LDA precisa de pelo menos 2.
if NUM_TOPICOS_LDA < 2:
    print(f"Aviso: Menos de 2 tópicos únicos encontrados no LLM (encontrados: {NUM_TOPICOS_LDA}).")
    print("Usando k=2 como padrão mínimo para o LDA rodar.")
    NUM_TOPICOS_LDA = 2 # Valor mínimo para o LDA

print(f"Treinando modelo LDA com k = {NUM_TOPICOS_LDA} tópicos...")

# 2. Treinar o modelo
# (passes=15 e iterations=100 são bons padrões, aumente se a coerência ficar baixa)
lda_model = models.LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=NUM_TOPICOS_LDA,
    random_state=42,  # Para reprodutibilidade
    passes=15,
    iterations=100
)

print("\nModelo LDA treinado. Top 10 palavras por tópico:")
# Mostrar os tópicos que o LDA encontrou
for idx, topic in lda_model.print_topics(num_topics=NUM_TOPICOS_LDA, num_words=10):
    print(f"Tópico {idx}: {topic}")

Treinando modelo LDA com k = 285 tópicos...



Modelo LDA treinado. Top 10 palavras por tópico:
Tópico 0: 0.064*"start" + 0.064*"guy" + 0.064*"increase" + 0.043*"voted" + 0.043*"total" + 0.021*"fell" + 0.021*"covid" + 0.021*"per" + 0.021*"border" + 0.021*"taliban"
Tópico 1: 0.000*"hell" + 0.000*"deserves" + 0.000*"discovered" + 0.000*"connection" + 0.000*"reid" + 0.000*"henry" + 0.000*"member" + 0.000*"spanier" + 0.000*"torres" + 0.000*"sofia"
Tópico 2: 0.000*"hell" + 0.000*"deserves" + 0.000*"discovered" + 0.000*"connection" + 0.000*"reid" + 0.000*"henry" + 0.000*"member" + 0.000*"spanier" + 0.000*"torres" + 0.000*"sofia"
Tópico 3: 0.000*"hell" + 0.000*"deserves" + 0.000*"discovered" + 0.000*"connection" + 0.000*"reid" + 0.000*"henry" + 0.000*"member" + 0.000*"spanier" + 0.000*"torres" + 0.000*"sofia"
Tópico 4: 0.000*"hell" + 0.000*"deserves" + 0.000*"discovered" + 0.000*"connection" + 0.000*"reid" + 0.000*"henry" + 0.000*"member" + 0.000*"spanier" + 0.000*"torres" + 0.000*"sofia"
Tópico 5: 0.082*"league" + 0.077*"man" + 0.050*"r

In [15]:
print("Calculando métricas para o modelo LDA...")

TOP_N_WORDS = 10 # Número de palavras para definir um tópico (padrão)

# 1. Extrair os tópicos do LDA (lista de listas de palavras)
lda_topics_list = []
for i in range(NUM_TOPICOS_LDA):
    # Pega os termos (ID da palavra, prob)
    top_words_tuples = lda_model.get_topic_terms(i, topn=TOP_N_WORDS)
    # Converte os IDs de volta para palavras
    topic_words = [dictionary[word_id] for word_id, prob in top_words_tuples]
    lda_topics_list.append(topic_words)
    
# 2. Calcular Coerência C_v
coherence_lda = calculate_topic_coherence(
    topics_with_words_list=lda_topics_list,
    documents_processed=processed_docs,
    dictionary=dictionary,
    coherence_type='c_v'
)

# 3. Calcular Diversidade
diversity_lda = calculate_topic_diversity(
    topics_with_words_list=lda_topics_list
)

print(f"\n--- Métricas Globais do LDA ---")
print(f"Coerência (C_v): {coherence_lda:.4f}")
print(f"Diversidade:     {diversity_lda:.4f}")

Calculando métricas para o modelo LDA...
Calculando Topic Coherence (c_v) para o LDA...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Coerência LDA (C_v) calculada: 0.4202673821553947
Calculando Topic Diversity para o LDA...
Diversidade LDA calculada: 0.3196491228070175

--- Métricas Globais do LDA ---
Coerência (C_v): 0.4203
Diversidade:     0.3196


In [16]:
print("Mapeando resultados do LDA de volta para o DataFrame...")

# 1. Criar um mapa de ID do tópico -> String de palavras
# (ex: 0 -> "game, team, score, play...")
topic_id_to_string_map = {}
for i, topic in enumerate(lda_topics_list):
    topic_id_to_string_map[i] = ", ".join(topic) # Junta as top 10 palavras

# 2. Encontrar o tópico dominante para cada documento
classificacoes_lda = []
for doc_bow in corpus:
    if not doc_bow:
        # Documento ficou vazio após pré-processamento
        classificacoes_lda.append("DOCUMENTO_VAZIO")
        continue
        
    # Pega a lista de (id_topico, probabilidade)
    topic_probs = lda_model.get_document_topics(doc_bow)
    
    if not topic_probs:
        # Modelo não conseguiu classificar (raro)
        classificacoes_lda.append("NAO_CLASSIFICADO")
        continue
    
    # Encontra o tópico com a maior probabilidade
    dominant_topic_id = max(topic_probs, key=lambda x: x[1])[0]
    
    # Usa o mapa para obter a string de palavras
    classificacoes_lda.append(topic_id_to_string_map[dominant_topic_id])

# 3. Adicionar as novas colunas ao DataFrame
df['classificacao_LDA'] = classificacoes_lda
df['topic_coherence_LDA'] = coherence_lda
df['topic_diversity_LDA'] = diversity_lda

print("Colunas do LDA adicionadas ao DataFrame.")
df.head()

Mapeando resultados do LDA de volta para o DataFrame...
Colunas do LDA adicionadas ao DataFrame.


,documento,classificação 1,classificação 2,topic_coherence,topic_diversity,classificacao_LDA,topic_coherence_LDA,topic_diversity_LDA
0,I love you #SEVENTEEN you icons #세븐틴 this won...,Music,K-Pop,0.869849,0.614853,"url, sign, change, via, stop, seventeen, clima...",0.420267,0.319649
1,Picking up some good stuff (@ Joe - {{USERNAME...,Shopping,Shopping,0.869849,0.614853,"new, url, album, username, via, opty, releases...",0.420267,0.319649
2,“I hadn’t quite understood the full extent of ...,Trade,Brexit \n\nExplanation:\n\nThe initial propose...,0.869849,0.614853,"miss, look, says, coffee, confused, clean, hus...",0.420267,0.319649
3,"Congrats to {{USERNAME}} , {{USERNAME}} , {{US...",Webpage,Social Media Promotion,0.869849,0.614853,"username, coming, new, later, video, glamberts...",0.420267,0.319649
4,"An excellent, a beautiful person not only with...",Music,Singers,0.869849,0.614853,"love, going, username, beautiful, stone, know,...",0.420267,0.319649


In [17]:
# --- Salvar Resultados Finais ---

# Reordenar colunas para melhor visualização (opcional)
colunas_llm = ['documento', 'classificação 1', 'classificação 2', 'topic_coherence', 'topic_diversity']
colunas_lda = ['classificacao_LDA', 'topic_coherence_LDA', 'topic_diversity_LDA']
df = df[colunas_llm + colunas_lda]

# Salvar o DataFrame final
OUTPUT_CSV_FILENAME = "resultados_completos_llm_lda.csv"
df.to_csv(OUTPUT_CSV_FILENAME, index=False, encoding='utf-8-sig')

print(f"\n--- PROCESSO CONCLUÍDO ---")
print(f"DataFrame final com resultados comparativos salvo em '{OUTPUT_CSV_FILENAME}'")

# Mostrar o resultado
df.head()


--- PROCESSO CONCLUÍDO ---
DataFrame final com resultados comparativos salvo em 'resultados_completos_llm_lda.csv'


,documento,classificação 1,classificação 2,topic_coherence,topic_diversity,classificacao_LDA,topic_coherence_LDA,topic_diversity_LDA
0,I love you #SEVENTEEN you icons #세븐틴 this won...,Music,K-Pop,0.869849,0.614853,"url, sign, change, via, stop, seventeen, clima...",0.420267,0.319649
1,Picking up some good stuff (@ Joe - {{USERNAME...,Shopping,Shopping,0.869849,0.614853,"new, url, album, username, via, opty, releases...",0.420267,0.319649
2,“I hadn’t quite understood the full extent of ...,Trade,Brexit \n\nExplanation:\n\nThe initial propose...,0.869849,0.614853,"miss, look, says, coffee, confused, clean, hus...",0.420267,0.319649
3,"Congrats to {{USERNAME}} , {{USERNAME}} , {{US...",Webpage,Social Media Promotion,0.869849,0.614853,"username, coming, new, later, video, glamberts...",0.420267,0.319649
4,"An excellent, a beautiful person not only with...",Music,Singers,0.869849,0.614853,"love, going, username, beautiful, stone, know,...",0.420267,0.319649
